In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import ast

FILE_NAME = "/work/dlclarge1/matusd-fsbench_plot/plotting_fs_benchmark/result_files/results_per_split_0505_late.csv"
PLOT_NAME = "elo_ranking_v1_1000"

# A "cell" is one (dataset, downstream model, budget, metric) tuple.
# Within a cell, all FS methods are directly comparable on the same metric scale.
CELL_KEYS = ["tid", "downstream_model", "feature_selection_budget_index", "metric"]

N_BOOTSTRAP = 1000
SCALE = 400 / np.log(10)   # logit units -> Elo units
ANCHOR = 1000              # rating to assign the mean method (cosmetic)
RNG = np.random.default_rng(42)

NAME_MAP = {
    "SequentialBackwardEliminationFeatureSelector": "RFE",
    "SequentialForwardSelectionFeatureSelector": "SFS",
    "AccuracyFeatureSelector": "LOCO",
    "ReliefFFeatureSelector": "(R)ReliefF",
    "ANOVAFFeatureSelector": "F-test",
}


def beautify(name):
    """Apply explicit renames; fall back to stripping the 'FeatureSelector' suffix."""
    if name in NAME_MAP:
        return NAME_MAP[name]
    return name.replace("FeatureSelector", "")


def beautify(name):
    """Apply explicit renames, then strip the 'FeatureSelector' suffix."""
    if name in NAME_MAP:
        return NAME_MAP[name]
    return name.replace("FeatureSelector", "")

In [2]:

def aggregate_to_cells(df):
    """Collapse splits/folds/repeats: one error value per (cell, FS method)."""
    return (
        df.dropna(subset=CELL_KEYS + ["feature_selection_method", "metric_error"])
          .groupby(CELL_KEYS + ["feature_selection_method"], as_index=False)["metric_error"]
          .mean()
          .rename(columns={"metric_error": "error"})
    )


def build_matches(cell_df, methods):
    """For each cell, generate all pairwise matches between FS methods.
    Returns arrays of (winner_idx, loser_idx, dataset_id) for cluster bootstrap."""
    method_to_idx = {m: i for i, m in enumerate(methods)}
    winners, losers, datasets = [], [], []

    for _, group in cell_df.groupby(CELL_KEYS):
        errs = group.set_index("feature_selection_method")["error"]
        ms = errs.index.tolist()
        for i in range(len(ms)):
            for j in range(i + 1, len(ms)):
                a, b = ms[i], ms[j]
                if errs[a] == errs[b]:
                    continue
                if errs[a] < errs[b]:
                    winners.append(method_to_idx[a]); losers.append(method_to_idx[b])
                else:
                    winners.append(method_to_idx[b]); losers.append(method_to_idx[a])
                datasets.append(group["tid"].iloc[0])

    return (
        np.array(winners, dtype=np.int64),
        np.array(losers, dtype=np.int64),
        np.array(datasets),
    )

def fit_bt(winners, losers, n_methods):
    """Bradley-Terry MLE via logistic regression with no intercept.
    Symmetrize the design: each match contributes two rows (winner=1, loser=0)
    so sklearn sees both classes."""
    M = len(winners)
    X = np.zeros((2 * M, n_methods))
    # Winner perspective: +1 winner, -1 loser, label 1
    X[np.arange(M), winners] = 1
    X[np.arange(M), losers] = -1
    # Loser perspective: -1 winner, +1 loser, label 0
    X[M + np.arange(M), winners] = -1
    X[M + np.arange(M), losers] = 1
    y = np.concatenate([np.ones(M), np.zeros(M)])

    model = LogisticRegression(fit_intercept=False, C=1e9, solver="lbfgs", max_iter=1000) # regularization to not send scores to inf if method is really good
    model.fit(X, y)
    return model.coef_.flatten()


def plot(methods, point_elo, lo, hi, out_path):
    order = np.argsort(point_elo)
    methods_sorted = [beautify(m) for m in np.array(methods)[order]]
    pt, lo, hi = point_elo[order], lo[order], hi[order]

    fig, ax = plt.subplots(figsize=(max(10, 0.7 * len(methods)), 5))
    x = np.arange(len(methods))

    bottom = np.floor(lo.min() / 50) * 50 - 25    
    ax.bar(
        x, pt - bottom, bottom=bottom, width=0.7,
        color="#0072B2", edgecolor="white", linewidth=0.5,
    )
    ax.errorbar(
        x, pt, yerr=[pt - lo, hi - pt],
        fmt="none", ecolor="black", capsize=3, linewidth=1,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(methods_sorted, rotation=30, ha="right")
    ax.set_ylabel("Elo")
    ax.set_ylim(bottom=bottom)
    ax.grid(axis="y", linestyle="-", alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig(str(out_path)+".png", dpi=300, bbox_inches="tight")
    plt.savefig(str(out_path)+".pdf", dpi=300, bbox_inches="tight")
    plt.close()

def extract_model_cls(s):
    try:
        return ast.literal_eval(s)["model_cls"]
    except Exception:
        return None

def main():
    COLUMNS = [
        "experiment_method_name_string", "model_details", "tid", "name",
        "fold", "repeat", "sample", "split_idx",
        "metric_error", "metric_error_val", "metric", "problem_type",
        "time_train_s", "time_infer_s",
        "feature_selection_method", "feature_selection_is_scoring_method",
        "selected_feature_names", "max_features",
        "feature_selection_budget_total", "feature_selection_budget_index",
        "feature_selection_fit_time", "feature_selection_time_limit",
    ]

    df = pd.read_csv(FILE_NAME, low_memory=False, header=None, names=COLUMNS)

    numeric_cols = ["metric_error", "metric_error_val", "time_train_s", "time_infer_s",
                    "max_features", "feature_selection_budget_total",
                    "feature_selection_budget_index", "feature_selection_fit_time",
                    "feature_selection_time_limit", "fold", "repeat", "sample", "split_idx"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["downstream_model"] = df["model_details"].apply(extract_model_cls)

    cell_df = aggregate_to_cells(df)
    methods = sorted(cell_df["feature_selection_method"].unique())

    winners, losers, datasets = build_matches(cell_df, methods)

    # Fit point estimate and bootstrap in raw logit space
    point_beta = fit_bt(winners, losers, len(methods))
    boot_betas = np.zeros((N_BOOTSTRAP, len(methods)))
    unique_ds = np.unique(datasets)
    for b in range(N_BOOTSTRAP):
        sampled = RNG.choice(unique_ds, size=len(unique_ds), replace=True)
        idx = np.concatenate([np.where(datasets == d)[0] for d in sampled])
        boot_betas[b] = fit_bt(winners[idx], losers[idx], len(methods))

    # Anchor consistently: pin RandomFeatureSelector to ANCHOR. now we can compare to random (one shared shift for all bootstraps)
    # uncertainty measure relative to random
    random_idx = methods.index("RandomFeatureSelector")
    shift = point_beta[random_idx]
    point_elo = (point_beta - shift) * SCALE + ANCHOR # convert to elo
    boot_elos = (boot_betas - shift) * SCALE + ANCHOR

    lo = np.percentile(boot_elos, 2.5, axis=0)
    hi = np.percentile(boot_elos, 97.5, axis=0)

    plot(methods, point_elo, lo, hi, Path(PLOT_NAME))
    print(f"✅ Saved to {PLOT_NAME}")

In [3]:

main()

FileNotFoundError: [Errno 2] No such file or directory: '/work/dlclarge1/matusd-fsbench_plot/plotting_fs_benchmark/result_files/results_per_split_0505_late.csv'